In [ ]:
# @title 1. Setup Sistem & Pilih Model Awal (Bootstrap + Preset Downloader)
import os, sys, shutil, subprocess
from pathlib import Path

# @markdown ### 📦 Sumber Repositori GitHub (Otomatis di-clone ke `/content/ai-studio-t4`)
REPO_URL = "https://github.com/your-org/ai-studio-t4.git" # @param {type:"string"}
STUDIO_DIR = Path("/content/ai-studio-t4") if Path("/content").exists() else Path.cwd()

# @markdown ---
# @markdown ### ☑️ Pilih Bundel Engine Awal yang Ingin Diunduh Sekarang
# @markdown *(Engine lain tetap bisa diunduh kapan saja secara On-Demand langsung dari dalam UI Gradio)*
INSTALL_ANIMA = True # @param {type:"boolean"}
INSTALL_Z_IMAGE_TURBO = False # @param {type:"boolean"}
INSTALL_FLUX_FUSION_V2 = False # @param {type:"boolean"}
INSTALL_MOSS_SFT = False # @param {type:"boolean"}
INSTALL_QWEN_21_TURBO = False # @param {type:"boolean"}
INSTALL_KREA2_GGUF = False # @param {type:"boolean"}
INSTALL_CHROMA_Q5 = False # @param {type:"boolean"}
INSTALL_WAN21_I2V = False # @param {type:"boolean"}
INSTALL_UPSCALERS = True # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🔑 Token Autentikasi Opsional (Untuk Gated Repo atau Civitai Early-Access)
HF_TOKEN = "" # @param {type:"string"}
CIVITAI_TOKEN = "" # @param {type:"string"}
MODELSCOPE_TOKEN = "" # @param {type:"string"}

os.environ["HF_TOKEN"] = HF_TOKEN.strip()
os.environ["CIVITAI_TOKEN"] = CIVITAI_TOKEN.strip()
os.environ["MODELSCOPE_TOKEN"] = MODELSCOPE_TOKEN.strip()

# 1. Clone atau gunakan folder lokal ai-studio-t4
if not (STUDIO_DIR / "bootstrap.py").exists():
    if Path("bootstrap.py").exists():
        STUDIO_DIR = Path.cwd()
    else:
        print(f"🔄 Meng-clone repositori dari {REPO_URL} ke {STUDIO_DIR}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(STUDIO_DIR)], check=True)

if str(STUDIO_DIR) not in sys.path:
    sys.path.insert(0, str(STUDIO_DIR))

# 2. Susun daftar engine yang dicentang
selected_engines = []
if INSTALL_ANIMA: selected_engines.append("anima")
if INSTALL_Z_IMAGE_TURBO: selected_engines.append("z_image")
if INSTALL_FLUX_FUSION_V2: selected_engines.append("flux_fusion")
if INSTALL_MOSS_SFT: selected_engines.append("moss")
if INSTALL_QWEN_21_TURBO: selected_engines.append("qwen21")
if INSTALL_KREA2_GGUF: selected_engines.append("krea2")
if INSTALL_CHROMA_Q5: selected_engines.append("chroma")
if INSTALL_WAN21_I2V: selected_engines.append("wan21")

# 3. Jalankan bootstrap dengan live log streaming ke output sel Colab
cmd = [
    sys.executable,
    "-u",
    str(STUDIO_DIR / "bootstrap.py"),
    "--engines",
    ",".join(selected_engines) if selected_engines else "none",
]
if INSTALL_UPSCALERS:
    cmd.append("--upscalers")
if HF_TOKEN.strip():
    cmd.extend(["--hf-token", HF_TOKEN.strip()])
if CIVITAI_TOKEN.strip():
    cmd.extend(["--civitai-token", CIVITAI_TOKEN.strip()])
if MODELSCOPE_TOKEN.strip():
    cmd.extend(["--modelscope-token", MODELSCOPE_TOKEN.strip()])

with subprocess.Popen(
    cmd,
    cwd=str(STUDIO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    retcode = proc.wait()
    if retcode != 0:
        raise RuntimeError(f"Bootstrap gagal dengan kode keluar {retcode}. Periksa baris log terakhir di atas.")

In [ ]:
# @title 2. (Opsional) Unduh Custom LoRA / Model via Link (Civitai · HF · ModelScope)
import os, sys
from pathlib import Path

STUDIO_DIR = Path("/content/ai-studio-t4") if Path("/content/ai-studio-t4").exists() else Path.cwd()
if str(STUDIO_DIR) not in sys.path:
    sys.path.insert(0, str(STUDIO_DIR))

# @markdown ### 🚀 Paste Link Custom LoRA / UNet (Pisahkan beberapa link dengan koma `,` atau `;`)
# @markdown *Mendukung link `HuggingFace`, `Civitai (.com/.red)`, `ModelScope`, atau `Git LFS repo`. Tambahkan `|nama_kustom.safetensors` di akhir link untuk rename otomatis.*
CUSTOM_LORA_URLS = "" # @param {type:"string"}
CUSTOM_UNET_URLS = "" # @param {type:"string"}

from core.downloader import UniversalDownloader
from bootstrap import print_storage_summary

downloader = UniversalDownloader()

for url_entry in UniversalDownloader.parse_entries(CUSTOM_LORA_URLS):
    downloader.download_custom_url(url_entry, target_subdir="loras")

for url_entry in UniversalDownloader.parse_entries(CUSTOM_UNET_URLS):
    downloader.download_custom_url(url_entry, target_subdir="diffusion_models")

print_storage_summary()

In [ ]:
# @title 3. Launch Fooocus-Style Multi-Engine Studio
import os, sys, subprocess
from pathlib import Path

STUDIO_DIR = Path("/content/ai-studio-t4") if Path("/content/ai-studio-t4").exists() else Path.cwd()
if str(STUDIO_DIR) not in sys.path:
    sys.path.insert(0, str(STUDIO_DIR))

# Izinkan Gradio membaca & memindahkan hasil render dari /content/Studio_Gallery dan /content/ComfyUI
os.environ["GRADIO_ALLOWED_PATHS"] = "/content,/content/Studio_Gallery,/content/ComfyUI,/content/ai-studio-t4,/tmp"

# @markdown ### 🔐 Opsi Keamanan Login Gradio (`https://xxxx.gradio.live`)
# @markdown *Jika `True`, sistem akan membuat password acak dan menampilkannya di log atas sebelum link dibuka.*
ENABLE_GRADIO_AUTH = False # @param {type:"boolean"}

cmd = [sys.executable, "-u", str(STUDIO_DIR / "run_studio.py"), "--share"]
if ENABLE_GRADIO_AUTH:
    cmd.append("--auth")

with subprocess.Popen(
    cmd,
    cwd=str(STUDIO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    retcode = proc.wait()
    if retcode != 0:
        raise RuntimeError(f"Studio berhenti dengan kode keluar {retcode}.")